# ストリーマシミュレーション Python版
Pythonカーネルで上から順に実行します。NumPy、FiPy、SciPy、Matplotlibが必要です。電位はFiPyのPoisson求解、電子輸送はFiPyの陽解法を使用します。終了時刻は`t_end`、保存間隔は`output_interval`で指定します。

配列添字は `[半径, 軸方向]`。電場はTd、密度はm⁻³、電位はVです。出力0番はt=0の初期状態で、電位・電場・密度は全て同じ時刻です。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'streamer_drift_diffusion').is_dir():
    ROOT = ROOT / 'pyaxistreamer-master'
if not (ROOT / 'streamer_drift_diffusion').is_dir():
    raise RuntimeError('streamer_python またはその親フォルダを作業ディレクトリにしてください')
sys.path.insert(0, str(ROOT))
from streamer_drift_diffusion import SimulationConfig, run_simulation, load_voltage_waveform, applied_voltage
import fipy
print('Python:', sys.executable)
print('FiPy:', fipy.__version__)

In [ ]:
# 電場・速度はスタガード配置。CSVのEx/Ey/absEはセル中心 [Td]。
# 計算条件は、このSimulationConfigにまとめて指定する。
config = SimulationConfig(
    nr=400, nz=300, delta=5e-6, electrode_gap=1e-3,
    electron_diffusion=0.0,  # 検証用の一定係数 [m^2/s]。0は拡散なし。
    voltage_mode='impulse', voltage=4000.0,
    impulse_peak_voltage=6000.0, impulse_time_scale=0.25,
    adaptive_timestep=True, t_end=5.001e-9,
    output_interval=1e-10, dt_max=1e-12,
    max_steps=20_000,
)
OUTPUT_DIR = ROOT / ('outputdata_adaptive_staggered' if config.adaptive_timestep else 'outputdata_staggered')
print(config)

In [ ]:
state = run_simulation(config, output_dir=OUTPUT_DIR)
print('CSV:', OUTPUT_DIR)
print('maximum electron density [m^-3]:', state.ne.max())

step = 104, Time = 0.100000 ns, dt = 5.000000e-13 s
step = 208, Time = 0.200000 ns, dt = 5.000000e-13 s
step = 312, Time = 0.300000 ns, dt = 5.000000e-13 s
step = 416, Time = 0.400000 ns, dt = 5.000000e-13 s
step = 520, Time = 0.500000 ns, dt = 5.000000e-13 s
step = 624, Time = 0.600000 ns, dt = 5.000000e-13 s
step = 728, Time = 0.700000 ns, dt = 5.000000e-13 s
step = 832, Time = 0.800000 ns, dt = 5.000000e-13 s
step = 936, Time = 0.900000 ns, dt = 5.000000e-13 s
step = 1040, Time = 1.000000 ns, dt = 5.000000e-13 s
step = 1145, Time = 1.100000 ns, dt = 1.077892e-14 s
step = 1257, Time = 1.200000 ns, dt = 4.648585e-13 s
step = 1369, Time = 1.300000 ns, dt = 5.775254e-13 s
step = 1481, Time = 1.400000 ns, dt = 5.700649e-13 s
step = 1593, Time = 1.500000 ns, dt = 6.028112e-13 s
step = 1707, Time = 1.600000 ns, dt = 3.395944e-13 s
step = 1844, Time = 1.700000 ns, dt = 6.255279e-13 s
step = 2072, Time = 1.800000 ns, dt = 1.721612e-13 s
step = 2667, Time = 1.900000 ns, dt = 4.575358e-14 s


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
fields = [(state.ne, 'Electron density [m$^{-3}$]'),
          (state.phi, 'Potential [V]'), (state.abs_e, '|E|/N [Td]')]
for ax, (values, title) in zip(axes, fields):
    image = ax.pcolormesh(state.rh * 1e3, state.zh * 1e3, values.T, shading='nearest', cmap='jet')
    ax.set(xlabel='r [mm]', ylabel='z [mm]', title=title)
    fig.colorbar(image, ax=ax)
plt.show()
# 必要な場合: fig.savefig(ROOT / 'final_state.png', dpi=160)

電位・電場は最終stepの密度更新前、電子密度は更新後の値です。Julia版の既存描画Notebookも、入力先をこの `OUTPUT_DIR` に変更して使用できます。